In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load the CSV Dataset

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/sem3/mini project - Bipolar Disorder/bipolar-raw-dataset.csv'
df = pd.read_csv(file_path)

display(df.head())

,Record ID,Gender:,Marital Status:,No of biological children:,Date of Assessment of File/ Interview:,Date of Registration:,Date of Birth:,Age,Birth City:,Birth State,...,Criteria B Score,Total Score,Kindly rate your social life and friendships before becoming ill,Kindly rate your self-Care and activities of daily living over the last 5 years,"In the last 5 years, were you going to-",Kindly rate your occupational functioning in the last 5 years.,Are you currently married or in a long-term relationship?,Kindly rate any impairment in capacity for intimate relationship in the last five years,"Any additional information pertaining to self care, work and social adjustment-",Genotyped?
0,2,Male,Married,2.0,2025-12-22,1996-01-17,1976-01-01,50.0,Kanakapura,Karnataka,...,2.0,7.0,3.0,1.0,Worked outside home,1.0,Yes,1.0,NaN,Yes
1,3,Male,Married,0.0,2023-02-07,1995-04-04,1956-01-01,67.1,BANGALORE,KARNATAKA,...,1.0,9.0,NaN,2.0,NaN,2.0,No,1.0,NaN,Yes
2,4,Male,Married,NaN,2023-02-13,1995-04-27,1975-01-01,48.1,NAMAKKAL,TAMIN NADU,...,0.0,10.0,NaN,4.0,Worked outside home,4.0,Yes,3.0,NaN,Yes
3,5,Male,Married,NaN,2023-02-14,1995-03-22,1965-01-01,58.1,MYSORE,KARNATAKA,...,2.0,8.0,NaN,4.0,NaN,NaN,Yes,3.0,NaN,Yes
4,6,Female,Married,1.0,2023-02-15,1995-04-07,1974-01-01,49.1,BANGALORE,KARNATAKA,...,4.0,6.0,3.0,5.0,NaN,4.0,Yes,3.0,NaN,Yes


## 2. Clean dataset column names

remove special characters like '(', ':', '=' etc...

transform " "(space) into underscores

In [ ]:
import re

def clean_column_names(dataframe: pd.DataFrame)-> pd.DataFrame:
  '''
  This function takes dataframe as input and return the dataframe with cleaned column names
  '''
  cleaned_cols = []
  for col in df.columns:
    c = col.strip().lower()
    c = re.sub(r'[^\w\s]', '', c)
    c = re.sub(r'\s+', '_', c)
    cleaned_cols.append(c)
  dataframe.columns = cleaned_cols # replacing the actual dataframe column names with new cleaned column names
  return dataframe

#call the function
df = clean_column_names(df)

print("Cleaned column count = ", len(df.columns))
print(df.columns.tolist())

Cleaned column count =  430
['record_id', 'gender', 'marital_status', 'no_of_biological_children', 'date_of_assessment_of_file_interview', 'date_of_registration', 'date_of_birth', 'age', 'birth_city', 'birth_state', 'birth_country', 'current_city', 'current_state', 'current_country', 'background', 'religion', 'income', 'education', 'years_of_education', 'occupation', 'occupation_details', 'detailed_workup', 'full_diagnosis', 'family_history_of_psychiatric_illness_in_first_degree_relatives_choice1_schizophreniapsychosis', 'family_history_of_psychiatric_illness_in_first_degree_relatives_choice2_bpad', 'family_history_of_psychiatric_illness_in_first_degree_relatives_choice3_rdd', 'family_history_of_psychiatric_illness_in_first_degree_relatives_choice4_substance_dependence', 'family_history_of_psychiatric_illness_in_first_degree_relatives_choice5_anxiety_disorders', 'fdr_scz_count', 'fdr_bpad_count', 'fdr_rdd_count', 'fdr_sud_count', 'fdr_anx_count', 'total_number_of_affected_first_degree_

## 3. Feature extraction

extracting core features from the 430 available features in the raw dataset


In [ ]:
selected_features = [
    # Demographics & Context
    'age', 'gender', 'marital_status', 'education',

    # Trajectory & Episode Metrics
    'age_at_onset_of_illness_first_episode',
    'total_number_of_episodes',
    'total_duration_of_illness_in_months',
    'total_duration_of_all_episodes_in_months',
    'total_duration_of_hypomanic_episodes_in_months',
    'total_number_of_manic_episodes',
    'total_number_of_hypomanic_episodes',
    'total_number_of_mixed_episodes',
    'total_number_of_depressive_episodes',
    'lifetime_history_of_rapid_cycling',
    'percentage_of_time_with_symptoms',
    'interepisodic_remission',
    'maximum_interepisodic_duration_in_months',
    'rate_time_spent_euthymic_since_onset',
    'rate_prominence_of_mixed_episodes',
    'in_first_five_years_of_illness_after_onset_rate_the_amount_of_time_spent_primarily_in_manic_vs_depressive_symptoms_on_5point_scale',
    'in_more_recent_years_of_illness_rate_the_amount_of_time_spent_primarily_in_manic_vs_depressive_symptoms_on_5point_scale',

    # Manic Symptom Domain
    'has_there_ever_been_a_manic_episode',
    'persistent_elevated_mood', 'persistent_irritability', 'dec_need_for_sleep',
    'inc_speech_rate_and_amount', 'foiracing_thoughts', 'distractibility',
    'inc_pma', 'grandiosity', 'overfamiliarity', 'increased_sexual_interest',
    'agitation', 'high_risk_behaviours',

    # Depressive Symptom Domain
    'has_there_ever_been_a_depressive_episode',
    'per_low_mood', 'emptiness', 'anh', 'ef', 'guilt', 'worthless',
    'poor_conc', 'self_harm_ideation', 'suicidal_attempt',

    # Psychotic & Negative Symptom Domain
    'has_there_ever_been_psychotic_symptoms', 'psy_symp',
    'del_ref', 'del_per', 'del_gran', 'del_sin', 'del_ctrl', 'del_soma',
    'thought_phen', 'ah', 'vh', 'disorg_speech', 'disorg_behaviour',
    'catatonia', 'avolition', 'blunted_affect', 'rate_negative_symptoms',

    # Severity & Functioning
    'insightinterepisode', 'social_dysfunction', 'relationship_dysfunction',
    'occupational_dysfunction', 'kindly_rate_your_occupational_functioning_in_the_last_5_years',
    'hospitalized', 'number_of_ip_care_recieved', 'a_criterion_score', 'criteria_b_score', 'total_score',

    # Comorbidities
    'psychiatric_comorbidity_choiceanxiety_disorders',
    'psychiatric_comorbidity_choicepersonality_disorders',
    'psychiatric_comorbidity_choiceocrds',
    'psychiatric_comorbidity_choicetrauma_and_related_disorders',
    'psychiatric_comorbidity_choiceadhd',
    'psychiatric_comorbidity_choiceeating_disorders',
    'comorbid_substance_use_disorder_choicetobacco_dependence',
    'comorbid_substance_use_disorder_choicealcohol_dependence',
    'comorbid_substance_use_disorder_choicecannabis_dependence'
]

df_subset = df[selected_features].copy()
print("Actual columns length in selected features list: ", len(selected_features))
print("Selected Subset Shape: ", df_subset.shape)

Actual columns length in selected features list:  80
Selected Subset Shape:  (367, 80)


In [ ]:
rename_dict = {
    'age_at_onset_of_illness_first_episode': 'age_onset',
    'has_there_ever_been_a_manic_episode': 'ever_manic',
    'has_there_ever_been_a_depressive_episode': 'ever_depressive',
    'has_there_ever_been_psychotic_symptoms': 'ever_psychotic',
    'foiracing_thoughts': 'racing_thoughts',
    'inc_pma': 'pma_mania',
    'per_low_mood': 'low_mood',
    'anh': 'anhedonia',
    'ef': 'executive_dysfunction',
    'poor_conc': 'poor_concentration',
    'kindly_rate_your_occupational_functioning_in_the_last_5_years': 'occupational_func_5yr',
    'in_first_five_years_of_illness_after_onset_rate_the_amount_of_time_spent_primarily_in_manic_vs_depressive_symptoms_on_5point_scale': 'mania_vs_dep_first5yr',
    'in_more_recent_years_of_illness_rate_the_amount_of_time_spent_primarily_in_manic_vs_depressive_symptoms_on_5point_scale': 'mania_vs_dep_recent',
    'psychiatric_comorbidity_choiceanxiety_disorders': 'comorbid_anxiety',
    'psychiatric_comorbidity_choicepersonality_disorders': 'comorbid_personality',
    'psychiatric_comorbidity_choiceocrds': 'comorbid_ocd',
    'psychiatric_comorbidity_choicetrauma_and_related_disorders': 'comorbid_trauma',
    'psychiatric_comorbidity_choiceadhd': 'comorbid_adhd',
    'psychiatric_comorbidity_choiceeating_disorders': 'comorbid_eating',
    'comorbid_substance_use_disorder_choicetobacco_dependence': 'substance_tobacco',
    'comorbid_substance_use_disorder_choicealcohol_dependence': 'substance_alcohol',
    'comorbid_substance_use_disorder_choicecannabis_dependence': 'substance_cannabis'
}

df_subset = df_subset.rename(columns=rename_dict)
df_subset.head()

,age,gender,marital_status,education,age_onset,total_number_of_episodes,total_duration_of_illness_in_months,total_duration_of_all_episodes_in_months,total_duration_of_hypomanic_episodes_in_months,total_number_of_manic_episodes,...,total_score,comorbid_anxiety,comorbid_personality,comorbid_ocd,comorbid_trauma,comorbid_adhd,comorbid_eating,substance_tobacco,substance_alcohol,substance_cannabis
0,50.0,Male,Married,8th std,17.0,11.0,395.6,41.5,2,7.0,...,7.0,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Checked,Unchecked,Unchecked
1,67.1,Male,Married,NaN,38.6,6.0,342.2,13.0,0,4.0,...,9.0,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked
2,48.1,Male,Married,"MSC, B.ED",20.3,7.0,333.6,7.0,0,6.0,...,10.0,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked
3,58.1,Male,Married,NOT LITERATE,30.2,6.0,334.7,7.0,0,4.0,...,8.0,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked
4,49.1,Female,Married,NaN,21.2,11.0,335.2,16.0,0,9.0,...,6.0,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked


## 4. Data cleaning

- analyse values of each feature
- handle missing values

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

missing_count = df_subset.isnull().sum()
# print(missing_count.to_dict())

missing_percent = (missing_count/len(df_subset))*100
# print(missing_percent)

dictionary = dict(missing_percent)
below20 = [key for key,value in dictionary.items() if value < 20]
btw20and40 = [key for key,value in dictionary.items() if value >= 20 and value < 40]
above40 = [key for key,value in dictionary.items() if value >= 40]

print(f"Missing Count below 20% {below20} \n Misssing count = {len(below20)}")
print(f"Missing Count between 20% and 40% {btw20and40} \n Misssing count = {len(btw20and40)}")
print(f"Missing Count above 40% {above40} \n Misssing count = {len(above40)}")



Missing Count below 20% ['age', 'gender', 'marital_status', 'total_number_of_episodes', 'total_duration_of_all_episodes_in_months', 'total_number_of_manic_episodes', 'lifetime_history_of_rapid_cycling', 'rate_time_spent_euthymic_since_onset', 'rate_prominence_of_mixed_episodes', 'mania_vs_dep_first5yr', 'mania_vs_dep_recent', 'persistent_elevated_mood', 'persistent_irritability', 'dec_need_for_sleep', 'inc_speech_rate_and_amount', 'racing_thoughts', 'distractibility', 'pma_mania', 'grandiosity', 'overfamiliarity', 'increased_sexual_interest', 'agitation', 'social_dysfunction', 'relationship_dysfunction', 'occupational_dysfunction', 'hospitalized', 'comorbid_anxiety', 'comorbid_personality', 'comorbid_ocd', 'comorbid_trauma', 'comorbid_adhd', 'comorbid_eating', 'substance_tobacco', 'substance_alcohol', 'substance_cannabis'] 
 Misssing count = 35
Missing Count between 20% and 40% ['education', 'age_onset', 'total_duration_of_illness_in_months', 'total_duration_of_hypomanic_episodes_in_mo

In [ ]:
numerical_cols = df_subset.select_dtypes(include=['int64', 'float64']).columns.to_list()
print(len(numerical_cols))
categorical_cols = df_subset.select_dtypes(include=['object', 'category']).columns.to_list()
print(len(categorical_cols))

21
59


In [ ]:
uniqsetofvaluesCat = {}
for col in categorical_cols:
  uniqsetofvaluesCat[col] = set(df_subset[col].to_list())
  print(f"{col} : {set(df_subset[col].to_list())}")


gender : {'Female', 'Male', nan}
marital_status : {'Separated', 'Married', 'Re-married', nan, 'Divorced', 'Not available', 'Widowed', 'Never married'}
education : {'ITI', '10th', '12th', '10th std', 'II nd PUC', '3rd std.', 'secondary', '7th Standard', '7th standard', 'Graduate', 'Primary ', '4th std', 'Bcom', '10TH STD', 'Ist PUC', 'II nd standard', 'Diploma In civil engineering', 'diploma student', '10 th class ', '5th class', 'BCom', '10th standard', 'diploma', '3rd Standard', '3', '8', 'Diploma in CS', 'NOT LITERATE', 'illiterate ', 'High school', 'Graduation', 'graduate ', 'BE', 'SSLC', '1st PUC', 'B.E.', '8th std', 'BA, DED, CET', 'Illiterate ', 'Diploma in software engineering', '5th Standard', '9th standard', '8th class ', '6th standard', 'Illiterate', 'SSLC fail', '9', 'puc', '4 th standard', 'Sslc', 'BA Sanskrit ', 'B.Com', '9th Standard', '5th standard', '9th std', '3RD STD', '2nd standard', '5 standard', 'Diploma Metallurgy engineering', '7th', 'Upto 7th Standard', '5th', '

In [ ]:
for col in categorical_cols:
  df_subset[col] = df_subset[col].fillna("Unknown")

In [ ]:
uniqsetofvaluesNum = {}
for col in numerical_cols:
  uniqsetofvaluesNum[col] = set(df_subset[col].to_list())
  print(f"{col} : {set(df_subset[col].to_list())}")

age : {nan, nan, 11.2, 15.2, 17.7, 18.6, 19.4, 20.9, 21.1, 22.0, 23.7, 24.2, 21.8, 26.5, 24.0, 28.7, 28.2, 30.7, 30.4, 32.5, 33.7, 34.4, 35.2, 36.7, 36.2, 37.1, 38.2, 39.2, 39.1, 41.2, 42.2, 43.2, 44.3, 45.2, 46.1, 47.1, 48.1, 49.1, 51.3, 50.0, 53.3, 54.5, 54.6, 55.7, 55.2, 55.3, 56.7, 57.7, 52.1, 55.1, 58.2, 58.1, 59.1, 61.1, 59.2, 59.6, 67.1, 62.9, 69.5, 67.5, 69.3, 74.2, 75.3, 76.1, 74.8, 75.0, 72.3, nan, nan, 74.5, 74.7, 84.0, 83.1, 17.4, nan, nan, 40.3, 19.0, nan, nan, nan, 21.4, 21.5, nan, nan, 21.0, 22.9, nan, 23.1, 23.5, 23.0, nan, 25.0, 25.1, 27.5, 46.0, 27.4, 46.5, 27.0, 28.6, 28.9, nan, 29.1, 29.9, 30.0, nan, nan, 30.9, 31.1, 31.6, nan, 31.0, 32.9, 32.1, 33.9, 33.6, nan, 39.6, 34.3, 34.9, nan, 34.8, 34.2, 35.9, 35.5, nan, 35.0, 35.3, 36.9, 36.1, 36.6, nan, 37.7, 37.2, 37.5, 37.4, 37.6, 38.0, 38.6, 38.1, 38.9, 38.8, 39.3, 39.8, nan, 39.0, nan, 40.7, 40.1, 40.4, 40.6, nan, 41.6, 41.7, 41.3, 40.9, 41.0, 42.6, 42.4, 42.9, 42.8, 42.0, 43.7, 43.1, 43.4, nan, 42.1, 44.6, 44.0, 44.8

In [ ]:
for col in numerical_cols:
  df_subset[col] = df_subset[col].fillna(df_subset[col].median())
